In [26]:
import os
import time
from pathlib import Path

import cv2
import pandas as pd
import torch
import torch.nn.functional as F
from natsort import natsorted
from ultralytics import YOLO

from code_programm.path import get_path_weight_model

In [27]:
print(torch.cuda.device_count())
print(torch.cuda.get_device_name())
model = YOLO(get_path_weight_model('best.pt'))

1
NVIDIA GeForce GTX 1080 Ti


In [28]:
wheel_ets_train_x = pd.DataFrame(columns=[i for i in range(96 * 128)])

In [29]:
folder_path = Path('D:\Dataset_for_autopilot')
files_and_folders = os.listdir(folder_path)

# Фильтруем только папки
folders = [f for f in files_and_folders if os.path.isdir(os.path.join(folder_path, f))]

# Сортируем папки по дате изменения
sorted_folders = sorted(folders, key=lambda x: os.path.getmtime(os.path.join(folder_path, x)), reverse=True)

# Выводим список папок
print("Папки в папке {} отсортированы по дате изменения: ".format(folder_path))
for folder in sorted_folders:
    print(os.path.join('D:\Dataset_for_autopilot', folder))

Папки в папке D:\Dataset_for_autopilot отсортированы по дате изменения: 
D:\Dataset_for_autopilot\2024-04-08 14-25-52
D:\Dataset_for_autopilot\2024-04-08 14-25-43
D:\Dataset_for_autopilot\2024-04-08 14-25-34
D:\Dataset_for_autopilot\2024-04-08 14-25-26
D:\Dataset_for_autopilot\2024-04-08 14-25-07
D:\Dataset_for_autopilot\2024-04-08 14-24-51
D:\Dataset_for_autopilot\2024-04-08 14-23-11
D:\Dataset_for_autopilot\2024-04-08 14-20-50
D:\Dataset_for_autopilot\2024-04-08 14-19-43
D:\Dataset_for_autopilot\2024-04-08 14-18-55
D:\Dataset_for_autopilot\2024-04-08 14-17-38
D:\Dataset_for_autopilot\2024-04-08 14-15-59
D:\Dataset_for_autopilot\2024-04-08 14-15-30
D:\Dataset_for_autopilot\2024-04-08 14-11-02
D:\Dataset_for_autopilot\2024-04-08 14-00-05
D:\Dataset_for_autopilot\new 2
D:\Dataset_for_autopilot\2024-04-02 19-12-38
D:\Dataset_for_autopilot\2024-04-02 19-12-21
D:\Dataset_for_autopilot\2024-04-02 19-11-55
D:\Dataset_for_autopilot\2024-04-02 19-09-18
D:\Dataset_for_autopilot\2024-04-02 19-07

In [30]:
paths = [
    # r'D:\Dataset_for_autopilot\2024-04-08 14-25-52',
    # r'D:\Dataset_for_autopilot\2024-04-08 14-25-43',
    # r'D:\Dataset_for_autopilot\2024-04-08 14-25-34',
    # r'D:\Dataset_for_autopilot\2024-04-08 14-25-26',
    # r'D:\Dataset_for_autopilot\2024-04-08 14-25-07',
    # r'D:\Dataset_for_autopilot\2024-04-08 14-24-51',
    # r'D:\Dataset_for_autopilot\2024-04-08 14-23-11',
    # r'D:\Dataset_for_autopilot\2024-04-08 14-20-50',
    # r'D:\Dataset_for_autopilot\2024-04-08 14-19-43',
    # r'D:\Dataset_for_autopilot\2024-04-08 14-18-55',
    # r'D:\Dataset_for_autopilot\2024-04-08 14-17-38',
    # r'D:\Dataset_for_autopilot\2024-04-08 14-15-59',
    # r'D:\Dataset_for_autopilot\2024-04-08 14-15-30',
    # r'D:\Dataset_for_autopilot\2024-04-08 14-11-02',
    r'D:\Dataset_for_autopilot\2024-04-08 14-00-05',
]

In [31]:
mass_results = []

for num_path in paths:
    path_i = os.path.join(num_path, f'road')
    if os.path.exists(f'{path_i}') and os.path.isdir(f'{path_i}'):
        png_files = [os.path.join(path_i, file) for file in os.listdir(path_i) if file.endswith('.png')]
        print("Полные пути к файлам в папке:")
        png_files = natsorted(png_files)
        print(png_files[0])
    else:
        print("Указанный путь не существует или не является папкой.")

    start_time = time.time()
    length = len(wheel_ets_train_x)

    combined_mask_old = torch.zeros((1, 1, 96, 128), device='cuda')
    combined_mask_new = torch.zeros((1, 1, 96, 128), device='cuda')

    for file in png_files:
        bgra_image = cv2.imread(file, cv2.IMREAD_UNCHANGED)
        bgr_image = cv2.cvtColor(bgra_image, cv2.COLOR_BGRA2BGR)
        results = model(bgr_image,
                        conf=0.3,
                        # show=True,
                        device='cuda',
                        verbose=False)
        if results[0].masks is not None:
            combined_mask_old = combined_mask_new * 0.4 + combined_mask_new * 0.6
            combined_mask_new.zero_()
            for i in results[0].masks.data:
                combined_mask_new += F.interpolate(i.unsqueeze(0).unsqueeze(0), size=(96, 128), mode='bilinear',
                                                   align_corners=True)

        combined_tensor = combined_mask_old + combined_mask_new

        wheel_ets_train_x.loc[len(wheel_ets_train_x)] = combined_tensor[0][0].flatten().detach().cpu().numpy()

    print('Изображений:', len(wheel_ets_train_x) - length, '\nСек:', time.time() - start_time, '\nCек\изображение:',
          (time.time() - start_time) / (len(wheel_ets_train_x) - length), '\n')
print('Всего:', len(wheel_ets_train_x))
cv2.destroyAllWindows()

Полные пути к файлам в папке:
D:\Dataset_for_autopilot\2024-04-08 14-00-05\road\2024-04-08 14-00-05_0.png
Изображений: 17051 
Сек: 2039.3403596878052 
Cек\изображение: 0.11960239045732246 

Всего: 17051


In [32]:
wheel_ets_train_x = wheel_ets_train_x.drop(wheel_ets_train_x.tail(1).index, axis = 0)

In [33]:
wheel_ets_train_x.to_csv(r'C:\PycharmProjects\ETS_Autopilot\dataset_for_wheel_nn\X_train_road_2_2.csv', index=False)

In [10]:
path_i = os.path.join(paths[0], f'road')
if os.path.exists(f'{path_i}') and os.path.isdir(f'{path_i}'):
    png_files = [os.path.join(path_i, file) for file in os.listdir(path_i) if file.endswith('.png')]
    png_files = natsorted(png_files)

In [11]:
combined_mask_old = torch.zeros((1, 1, 96, 128), device='cuda')
combined_mask_new = torch.zeros((1, 1, 96, 128), device='cuda')

In [12]:
for i, ii in enumerate(png_files):
    bgra_image = cv2.imread(ii, cv2.IMREAD_UNCHANGED)
    bgr_image = cv2.cvtColor(bgra_image, cv2.COLOR_BGRA2BGR)
    results = model(bgr_image,
                    conf=0.6,
                    device='cuda',
                    verbose=False,
                    # show=True
                    )

    if results[0].masks is not None:
        combined_mask_old = combined_mask_old * 0.4 + combined_mask_new * 0.6
        combined_mask_new.zero_()
        for i in results[0].masks.data:
            combined_mask_new += F.interpolate(i.unsqueeze(0).unsqueeze(0), size=(96, 128), mode='bilinear',
                                               align_corners=True)

        combined_tensor = combined_mask_old + combined_mask_new

        cv2.imshow('mask', combined_tensor[0][0].cpu().detach().numpy())
        cv2.waitKey()
        print(combined_tensor[0][0].flatten().detach().cpu().numpy())
        print(len(combined_tensor[0][0].flatten().detach().cpu().numpy()))
cv2.destroyAllWindows()

[          0           0           0 ...           0           0           0]
12288
[          0           0           0 ...           0           0           0]
12288
[          0           0           0 ...           0           0           0]
12288
[          0           0           0 ...           0           0           0]
12288
[          0           0           0 ...           0           0           0]
12288
[          0           0           0 ...           0           0           0]
12288
[          0           0           0 ...           0           0           0]
12288
[          0           0           0 ...           0           0           0]
12288
[          0           0           0 ...           0           0           0]
12288
[          0           0           0 ...           0           0           0]
12288
[          0           0           0 ...           0           0           0]
12288
[          0           0           0 ...           0           0           0

In [13]:
cv2.destroyAllWindows()